In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import zipfile
from collections import Counter
import os

zip_path = "/content/drive/My Drive/VeriCure_Project/dataset_complete.zip"

if not os.path.exists(zip_path):
    print(f"[!] File not found at: {zip_path}")
else:
    print("=========================================================")
    print("      VERICURE: DATASET INTEGRITY REPORT                 ")
    print("=========================================================\n")

    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Get the list of all file paths inside the zip archive
        all_files = zip_ref.namelist()

        # Extract the top-level directory/class names
        class_counts = Counter()
        for file in all_files:
            # Filter for images to avoid counting directory markers
            if file.lower().endswith(('.jpg', '.jpeg', '.png')):
                parts = file.split('/')
                if len(parts) > 1:
                    class_name = parts[0]  # If files are zipped inside a root folder
                    if class_name == "dataset_complete" and len(parts) > 2:
                        class_name = parts[1]
                    class_counts[class_name] += 1

        # Print the results in a scannable format
        total_images = 0
        print(f"{'Class Directory Name':<30} | {'Image Count':<12}")
        print("-" * 45)
        for class_name, count in sorted(class_counts.items()):
            print(f"{class_name:<30} | {count:<12}")
            total_images += count

        print("-" * 45)
        print(f"{'TOTAL IMAGES IN ARCHIVE':<30} | {total_images:<12}")
        print("=========================================================")

      VERICURE: DATASET INTEGRITY REPORT                 

Class Directory Name           | Image Count 
---------------------------------------------
Background_Unknown             | 500         
froben_Counterfeit             | 500         
froben_Genuine                 | 500         
glucophage_Counterfeit         | 500         
glucophage_Genuine             | 500         
nise_Counterfeit               | 500         
nise_Genuine                   | 500         
panadol_Counterfeit            | 500         
panadol_Genuine                | 500         
---------------------------------------------
TOTAL IMAGES IN ARCHIVE        | 4500        


In [1]:
import os
import zipfile
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# =====================================================================
# HARDWARE CONFIGURATION & PIPELINE TUNING (RAM OPTIMIZED)
# =====================================================================
ZIP_PATH = "/content/drive/My Drive/VeriCure_Project/dataset_complete.zip"
DRIVE_DIR = "/content/drive/My Drive/VeriCure_Project"
LOCAL_EXTRACT_DIR = "dataset_runtime"

MODEL_VERSION = "v1.0"
MODEL_NAME = f"VeriCure_Model_{MODEL_VERSION}.keras"
MODEL_SAVE_PATH = os.path.join(DRIVE_DIR, MODEL_NAME)

IMG_SIZE = (512, 512)
BATCH_SIZE = 16
EPOCHS = 30
NUM_CLASSES = 9

def unpack_dataset():
    """Extracts the dataset locally onto Colab NVMe storage."""
    if os.path.exists(LOCAL_EXTRACT_DIR):
        shutil.rmtree(LOCAL_EXTRACT_DIR)

    print("[*] Extracting dataset archive onto local runtime storage...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_DIR)
    print("[+] Extraction complete.")

def build_data_pipelines():
    """Generates streaming data pipelines optimized to prevent RAM bloat."""
    print("\n[*] Initializing streaming data pipelines...")

    train_ds = tf.keras.utils.image_dataset_from_directory(
        LOCAL_EXTRACT_DIR,
        validation_split=0.2,
        subset="training",
        seed=42,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        LOCAL_EXTRACT_DIR,
        validation_split=0.2,
        subset="validation",
        seed=42,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    # REMOVED .cache() to prevent Colab Host RAM exhaustion.
    # We use prefetch to load batches asynchronously onto GPU VRAM instead.
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

    return train_ds, val_ds

def compile_vericure_net():
    """Compiles the convolutional neural network backbone."""
    print("[*] Assembling EfficientNet-B0 network topology...")

    img_augmentation = models.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.1),
        layers.RandomContrast(0.1)
    ], name="gpu_augmentation_stage")

    inputs = layers.Input(shape=(512, 512, 3), name="input_tensor")
    x = img_augmentation(inputs)

    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    base_model.trainable = False

    x = layers.GlobalAveragePooling2D(name="global_pooling_layer")(base_model.output)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4, name="dropout_regularization")(x)

    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="classification_output")(x)

    model = models.Model(inputs, outputs, name="VeriCure_EfficientNetB0")

    model.compile(
        optimizer=optimizers.Adam(learning_rate=1e-3),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def execute_training_pipeline():
    # Make sure Drive directory exists
    if not os.path.exists(DRIVE_DIR):
        print(f"[!] Drive directory not found. Did you run the Mount Drive cell?")
        return

    unpack_dataset()
    train_ds, val_ds = build_data_pipelines()
    model = compile_vericure_net()

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True, verbose=1),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.1, patience=2, verbose=1)
    ]

    print("\n=========================================================")
    print("      COMMENCING WEIGHT OPTIMIZATION ROUTINE            ")
    print("=========================================================\n")

    history = model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks
    )

    print("\n=======================================================================")
    print(f"[SUCCESS] Training complete. {MODEL_NAME} safely mirrored to Drive.")
    print("=======================================================================")

if __name__ == "__main__":
    execute_training_pipeline()

[*] Extracting dataset archive onto local runtime storage...
[+] Extraction complete.

[*] Initializing streaming data pipelines...
Found 4500 files belonging to 9 classes.
Using 3600 files for training.
Found 4500 files belonging to 9 classes.
Using 900 files for validation.
[*] Assembling EfficientNet-B0 network topology...

      COMMENCING WEIGHT OPTIMIZATION ROUTINE            

Epoch 1/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 188ms/step - accuracy: 0.4454 - loss: 1.5751
Epoch 1: val_accuracy improved from None to 0.69556, saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_Model_v1.0.keras

Epoch 1: finished saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_Model_v1.0.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 73s 255ms/step - accuracy: 0.5503 - loss: 1.1446 - val_accuracy: 0.6956 - val_loss: 0.6887 - learning_rate: 0.0010
Epoch 2/30
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.6618 - loss: 0.7929
Epoch 2: val_accuracy improved from 0.69556 to 0.71333,

In [2]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os

# =====================================================================
# CONFIGURATIONS FOR FINE-TUNING
# =====================================================================
DRIVE_DIR = "/content/drive/My Drive/VeriCure_Project"
BASE_MODEL_PATH = os.path.join(DRIVE_DIR, "VeriCure_Model_v1.0.keras")
FT_MODEL_SAVE_PATH = os.path.join(DRIVE_DIR, "VeriCure_Model_v1.1_FineTuned.keras")

# Re-build the pipelines (since runtime might have cleared variables)
train_ds, val_ds = build_data_pipelines()

print("\n[*] Loading baseline production weights...")
model = tf.keras.models.load_model(BASE_MODEL_PATH)

# =====================================================================
# UNFREEZING TOP EFFICIENTNET LAYERS
# =====================================================================
print("[*] Unfreezing top backbone convolutional blocks for fine-tuning...")

# Find the EfficientNet layer in our model container
for layer in model.layers:
    if "efficientnet" in layer.name:
        layer.trainable = True
        # Freeze all layers except the last 20 layers of the backbone
        for sub_layer in layer.layers[:-20]:
            sub_layer.trainable = False
        for sub_layer in layer.layers[-20:]:
            sub_layer.trainable = True

# We use an extremely small learning rate so we don't destroy pre-trained features
opt = tf.keras.optimizers.Adam(learning_rate=1e-5)

model.compile(
    optimizer=opt,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

model.summary()

# =====================================================================
# RUNNING FINE-TUNING PIPELINE
# =====================================================================
callbacks = [
    # Slightly higher patience for fine-tuning
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(FT_MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1)
]

print("\n=========================================================")
# This phase is usually incredibly fast and yields major accuracy jumps!
print("      COMMENCING BACKBONE FINE-TUNING STAGE             ")
print("=========================================================\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,  # Usually takes fewer epochs to converge during fine-tuning
    callbacks=callbacks
)

print("\n=======================================================================")
print(f"[SUCCESS] Fine-tuning complete. Best weights saved to: {FT_MODEL_SAVE_PATH}")
print("=======================================================================")


[*] Initializing streaming data pipelines...
Found 4500 files belonging to 9 classes.
Using 3600 files for training.
Found 4500 files belonging to 9 classes.
Using 900 files for validation.

[*] Loading baseline production weights...
[*] Unfreezing top backbone convolutional blocks for fine-tuning...


Model: "VeriCure_EfficientNetB0"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_tensor        │ (None, 512, 512,  │          0 │ -                 │
│ (InputLayer)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gpu_augmentation_s… │ (None, 512, 512,  │          0 │ input_tensor[0][… │
│ (Sequential)        │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling           │ (None, 512, 512,  │          0 │ gpu_augmentation… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ normalization       │ (None, 512, 512,  │          7 │ rescaling[0][0]   │
│ (Normalization)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ rescaling_1         │ (None, 512, 512,  │          0 │ normalization[0]… │
│ (Rescaling)         │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv_pad       │ (None, 513, 513,  │          0 │ rescaling_1[0][0] │
│ (ZeroPadding2D)     │ 3)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_conv (Conv2D)  │ (None, 256, 256,  │        864 │ stem_conv_pad[0]… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_bn             │ (None, 256, 256,  │        128 │ stem_conv[0][0]   │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stem_activation     │ (None, 256, 256,  │          0 │ stem_bn[0][0]     │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_dwconv      │ (None, 256, 256,  │        288 │ stem_activation[… │
│ (DepthwiseConv2D)   │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_bn          │ (None, 256, 256,  │        128 │ block1a_dwconv[0… │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_activation  │ (None, 256, 256,  │          0 │ block1a_bn[0][0]  │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_squeeze  │ (None, 32)        │          0 │ block1a_activati… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reshape  │ (None, 1, 1, 32)  │          0 │ block1a_se_squee… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_reduce   │ (None, 1, 1, 8)   │        264 │ block1a_se_resha… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_expand   │ (None, 1, 1, 32)  │        288 │ block1a_se_reduc… │
│ (Conv2D)            │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ block1a_se_excite   │ (None, 256, 256,  │          0 │ block1a_activati

 Total params: 4,066,220 (15.51 MB)

 Trainable params: 14,089 (55.04 KB)

 Non-trainable params: 4,052,131 (15.46 MB)


      COMMENCING BACKBONE FINE-TUNING STAGE             

Epoch 1/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.7044 - loss: 0.6326
Epoch 1: val_accuracy improved from None to 0.74111, saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_Model_v1.1_FineTuned.keras

Epoch 1: finished saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_Model_v1.1_FineTuned.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 71s 267ms/step - accuracy: 0.7047 - loss: 0.6336 - val_accuracy: 0.7411 - val_loss: 0.6352
Epoch 2/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 202ms/step - accuracy: 0.7189 - loss: 0.5890
Epoch 2: val_accuracy did not improve from 0.74111
225/225 ━━━━━━━━━━━━━━━━━━━━ 78s 249ms/step - accuracy: 0.7083 - loss: 0.6145 - val_accuracy: 0.7411 - val_loss: 0.6317
Epoch 3/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 199ms/step - accuracy: 0.7286 - loss: 0.5993
Epoch 3: val_accuracy did not improve from 0.74111
225/225 ━━━━━━━━━━━━━━━━━━━━ 55s 245ms/step - accuracy: 0.7258 - loss: 0.6003

In [3]:
import tensorflow as tf
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
import os

# =====================================================================
# CONFIGURATIONS FOR DEEP FINE-TUNING
# =====================================================================
DRIVE_DIR = "/content/drive/My Drive/VeriCure_Project"
BASE_MODEL_PATH = os.path.join(DRIVE_DIR, "VeriCure_Model_v1.0.keras")
FT_MODEL_SAVE_PATH = os.path.join(DRIVE_DIR, "VeriCure_Model_v1.1_DeepFineTuned.keras")

# Re-build dataset streams if Colab runtime refreshed
train_ds, val_ds = build_data_pipelines()

print("\n[*] Loading baseline production weights...")
model = tf.keras.models.load_model(BASE_MODEL_PATH)

# =====================================================================
# UNFREEZING THE ENTIRE BACKBONE
# =====================================================================
print("[*] Unfreezing ALL layers of the EfficientNet backbone...")

# Find the EfficientNet layer and unfreeze every single parameter
for layer in model.layers:
    if "efficientnet" in layer.name:
        layer.trainable = True
        print(f"    -> Successfully unfrozen: {layer.name}")

# Standard fine-tuning learning rate to break the 75% wall
opt = tf.keras.optimizers.Adam(learning_rate=1e-4)

model.compile(
    optimizer=opt,
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# =====================================================================
# RUNNING PIPELINE
# =====================================================================
callbacks = [
    # Give the model plenty of room to find its path
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(FT_MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, verbose=1)
]

print("\n=========================================================")
print("      COMMENCING DEEP CONVOLUTIONAL FINE-TUNING          ")
print("=========================================================\n")

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks
)

print("\n=======================================================================")
print(f"[SUCCESS] Deep fine-tuning complete!")
print(f"BEST MODEL PATH: {FT_MODEL_SAVE_PATH}")
print("=======================================================================")


[*] Initializing streaming data pipelines...
Found 4500 files belonging to 9 classes.
Using 3600 files for training.
Found 4500 files belonging to 9 classes.
Using 900 files for validation.

[*] Loading baseline production weights...
[*] Unfreezing ALL layers of the EfficientNet backbone...

      COMMENCING DEEP CONVOLUTIONAL FINE-TUNING          

Epoch 1/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 201ms/step - accuracy: 0.7090 - loss: 0.6333
Epoch 1: val_accuracy improved from None to 0.73889, saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_Model_v1.1_DeepFineTuned.keras

Epoch 1: finished saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_Model_v1.1_DeepFineTuned.keras
225/225 ━━━━━━━━━━━━━━━━━━━━ 71s 268ms/step - accuracy: 0.7086 - loss: 0.6227 - val_accuracy: 0.7389 - val_loss: 0.6560 - learning_rate: 1.0000e-04
Epoch 2/15
225/225 ━━━━━━━━━━━━━━━━━━━━ 0s 200ms/step - accuracy: 0.7178 - loss: 0.5887
Epoch 2: val_accuracy improved from 0.73889 to 0.74778, s

In [4]:
import os
import zipfile
import shutil
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# =====================================================================
# CONFIGURATION (UPGRADED TO V2.0)
# =====================================================================
ZIP_PATH = "/content/drive/My Drive/VeriCure_Project/dataset_complete.zip"
DRIVE_DIR = "/content/drive/My Drive/VeriCure_Project"
LOCAL_EXTRACT_DIR = "dataset_runtime"

MODEL_VERSION = "v2.0_HighCapacity"
MODEL_NAME = f"VeriCure_Model_{MODEL_VERSION}.keras"
MODEL_SAVE_PATH = os.path.join(DRIVE_DIR, MODEL_NAME)

# Scaling to native 224x224 input size drastically improves feature extraction speed & accuracy
IMG_SIZE = (224, 224)
BATCH_SIZE = 32  # Larger batch size stabilizes gradients
EPOCHS = 35
NUM_CLASSES = 9

def unpack_dataset():
    if not os.path.exists(LOCAL_EXTRACT_DIR):
        print("[*] Extracting dataset archive...")
        with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
            zip_ref.extractall(LOCAL_EXTRACT_DIR)
        print("[+] Extraction complete.")

def build_data_pipelines():
    print("\n[*] Initializing high-speed streaming data pipelines...")
    train_ds = tf.keras.utils.image_dataset_from_directory(
        LOCAL_EXTRACT_DIR,
        validation_split=0.2,
        subset="training",
        seed=42,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    val_ds = tf.keras.utils.image_dataset_from_directory(
        LOCAL_EXTRACT_DIR,
        validation_split=0.2,
        subset="validation",
        seed=42,
        image_size=IMG_SIZE,
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.prefetch(buffer_size=AUTOTUNE)

    return train_ds, val_ds

# =====================================================================
# MODEL ARCHITECTURE (HIGH-CAPACITY CLASSIFIER HEAD)
# =====================================================================

def compile_upgraded_vericure_net():
    print("[*] Building Upgraded High-Capacity Model...")

    img_augmentation = models.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.15),
        layers.RandomContrast(0.15),
        layers.RandomTranslation(height_factor=0.1, width_factor=0.1)
    ], name="gpu_augmentation_stage")

    inputs = layers.Input(shape=(224, 224, 3), name="input_tensor")
    x = img_augmentation(inputs)

    # We use a completely unfrozen backbone from the start with a gentle learning rate
    base_model = tf.keras.applications.EfficientNetB0(
        include_top=False,
        weights="imagenet",
        input_tensor=x
    )
    base_model.trainable = True

    x = layers.GlobalAveragePooling2D()(base_model.output)
    x = layers.BatchNormalization()(x)

    # --- ADDED DEEP DENSE BLOCK FOR FINE-GRAINED DETECTION ---
    x = layers.Dense(256, activation="relu", name="dense_reasoning_1")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)

    x = layers.Dense(128, activation="relu", name="dense_reasoning_2")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    # ---------------------------------------------------------

    outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="classification_output")(x)

    model = models.Model(inputs, outputs, name="VeriCure_Model_v2")

    # Gentle starting learning rate for the entire unfrozen network
    model.compile(
        optimizer=optimizers.Adam(learning_rate=2e-4),
        loss="categorical_crossentropy",
        metrics=["accuracy"]
    )
    return model

def run_upgraded_training():
    unpack_dataset()
    train_ds, val_ds = build_data_pipelines()
    model = compile_upgraded_vericure_net()

    callbacks = [
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True, verbose=1),
        ModelCheckpoint(MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
        ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, verbose=1)
    ]

    print("\n=========================================================")
    print("      TRAINING V2.0 MODEL (HIGH-CAPACITY HEAD)           ")
    print("=========================================================\n")

    model.fit(
        train_ds,
        validation_data=val_ds,
        epochs=EPOCHS,
        callbacks=callbacks
    )

if __name__ == "__main__":
    run_upgraded_training()


[*] Initializing high-speed streaming data pipelines...
Found 4500 files belonging to 9 classes.
Using 3600 files for training.
Found 4500 files belonging to 9 classes.
Using 900 files for validation.
[*] Building Upgraded High-Capacity Model...

      TRAINING V2.0 MODEL (HIGH-CAPACITY HEAD)           

Epoch 1/35
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 383ms/step - accuracy: 0.3015 - loss: 2.1915
Epoch 1: val_accuracy improved from None to 0.54111, saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_Model_v2.0_HighCapacity.keras

Epoch 1: finished saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_Model_v2.0_HighCapacity.keras
113/113 ━━━━━━━━━━━━━━━━━━━━ 96s 453ms/step - accuracy: 0.4164 - loss: 1.6806 - val_accuracy: 0.5411 - val_loss: 1.1700 - learning_rate: 2.0000e-04
Epoch 2/35
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 380ms/step - accuracy: 0.5763 - loss: 0.9940
Epoch 2: val_accuracy improved from 0.54111 to 0.57778, saving model to /content/drive/My Drive/VeriCure_

KeyboardInterrupt: 

In [6]:
import os
import zipfile
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# =====================================================================
# CONFIGURATION
# =====================================================================
ZIP_PATH = "/content/drive/My Drive/VeriCure_Project/dataset_complete.zip"
DRIVE_DIR = "/content/drive/My Drive/VeriCure_Project"
LOCAL_EXTRACT_DIR = "dataset_runtime"

MODEL_NAME = "VeriCure_EfficientNet_Boosted.keras"
MODEL_SAVE_PATH = os.path.join(DRIVE_DIR, MODEL_NAME)

IMG_SIZE = (224, 224)
BATCH_SIZE = 32
NUM_CLASSES = 9

if not os.path.exists(LOCAL_EXTRACT_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_EXTRACT_DIR)

train_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_EXTRACT_DIR, validation_split=0.2, subset="training", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
).prefetch(tf.data.AUTOTUNE)

val_ds = tf.keras.utils.image_dataset_from_directory(
    LOCAL_EXTRACT_DIR, validation_split=0.2, subset="validation", seed=42,
    image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="categorical"
).prefetch(tf.data.AUTOTUNE)

# =====================================================================
# MODEL ARCHITECTURE
# =====================================================================
img_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.15),
    layers.RandomContrast(0.15),
], name="gpu_augmentation_stage")

inputs = layers.Input(shape=(224, 224, 3), name="input_tensor")
x = img_augmentation(inputs)

base_model = tf.keras.applications.EfficientNetB0(
    include_top=False, weights="imagenet", input_tensor=x
)

# STAGE 1 FREEZE
base_model.trainable = False

x = layers.GlobalAveragePooling2D()(base_model.output)
x = layers.BatchNormalization()(x)

# High-capacity classifier head
x = layers.Dense(256, activation="relu", name="dense_reasoning_1")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.4)(x)

x = layers.Dense(128, activation="relu", name="dense_reasoning_2")(x)
x = layers.BatchNormalization()(x)
x = layers.Dropout(0.3)(x)

outputs = layers.Dense(NUM_CLASSES, activation="softmax", name="classification_output")(x)
model = models.Model(inputs, outputs, name="VeriCure_Boosted_Net")

# Compile Stage 1
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

# =====================================================================
# STAGE 1: QUICK WARM-UP (2 Epochs)
# =====================================================================
print("\n[*] Running 2-epoch head warm-up...")
model.fit(train_ds, validation_data=val_ds, epochs=2, verbose=1)

# =====================================================================
# STAGE 2: UNFREEZE WITH BOOSTED LEARNING RATE (2e-4)
# =====================================================================
print("\n[*] Unfreezing backbone with BOOSTED Fine-Tuning rate (2e-4)...")
base_model.trainable = True

model.compile(
    optimizer=optimizers.Adam(learning_rate=2e-4), # Boosted rate for fast convergence
    loss="categorical_crossentropy",
    metrics=["accuracy"]
)

callbacks = [
    EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
    ModelCheckpoint(MODEL_SAVE_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor="val_loss", factor=0.2, patience=2, verbose=1)
]

model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=15,
    callbacks=callbacks,
    verbose=1
)

print("\n=======================================================================")
print(f"[SUCCESS] Boosted training complete. Model saved to: {MODEL_SAVE_PATH}")
print("=======================================================================")

Found 4500 files belonging to 9 classes.
Using 3600 files for training.
Found 4500 files belonging to 9 classes.
Using 900 files for validation.

[*] Running 2-epoch head warm-up...
Epoch 1/2
113/113 ━━━━━━━━━━━━━━━━━━━━ 28s 137ms/step - accuracy: 0.4997 - loss: 1.2300 - val_accuracy: 0.5700 - val_loss: 0.8011
Epoch 2/2
113/113 ━━━━━━━━━━━━━━━━━━━━ 13s 117ms/step - accuracy: 0.5886 - loss: 0.8170 - val_accuracy: 0.5822 - val_loss: 0.6976

[*] Unfreezing backbone with BOOSTED Fine-Tuning rate (2e-4)...
Epoch 1/15
113/113 ━━━━━━━━━━━━━━━━━━━━ 0s 376ms/step - accuracy: 0.5222 - loss: 1.0132
Epoch 1: val_accuracy improved from None to 0.61444, saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_EfficientNet_Boosted.keras

Epoch 1: finished saving model to /content/drive/My Drive/VeriCure_Project/VeriCure_EfficientNet_Boosted.keras
113/113 ━━━━━━━━━━━━━━━━━━━━ 93s 441ms/step - accuracy: 0.5539 - loss: 0.8662 - val_accuracy: 0.6144 - val_loss: 0.6657 - learning_rate: 2.0000e-04